# Part 1i: Hyperparameter Tuning with Keras Tuner

**Objective:** Use Keras Tuner to automatically search for optimal hyperparameters.

---

In [1]:
!pip install keras-tuner -q
import numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt
print(f"Keras Tuner version: {kt.__version__}")

(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()
X_train, X_test = X_train.astype("float32")/255.0, X_test.astype("float32")/255.0
y_train, y_test = y_train.flatten(), y_test.flatten()
X_train_flat, X_test_flat = X_train.reshape(len(X_train),-1), X_test.reshape(len(X_test),-1)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.9 MB/s eta 0:00:00
Keras Tuner version: 1.4.8
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


## Define the Model-Building Function
Keras Tuner needs a function that takes an `hp` (hyperparameters) object and returns a compiled model.

In [2]:
def build_model(hp):
    """Build model with tunable hyperparameters."""
    model = keras.Sequential()
    model.add(layers.Input(shape=(3072,)))

    # Tune number of layers (1 to 4)
    for i in range(hp.Int('num_layers', 1, 4)):
        # Tune units per layer
        units = hp.Choice(f'units_{i}', values=[64, 128, 256, 512])
        model.add(layers.Dense(units, activation='relu'))

        # Tune whether to use batch normalization
        if hp.Boolean('use_batchnorm'):
            model.add(layers.BatchNormalization())

        # Tune dropout rate
        dropout_rate = hp.Float('dropout', min_value=0.0, max_value=0.5, step=0.1)
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(10, activation='softmax'))

    # Tune learning rate
    lr = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Quick test
test_model = build_model(kt.HyperParameters())
test_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │       196,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 197,322 (770.79 KB)

 Trainable params: 197,322 (770.79 KB)

 Non-trainable params: 0 (0.00 B)

## Run RandomSearch Tuner
We'll try RandomSearch — it samples hyperparameter combinations randomly, which is often more efficient than grid search.

In [3]:
# RandomSearch tuner
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,          # try 10 combinations
    executions_per_trial=1,  # 1 training run per combo
    directory='tuner_results',
    project_name='cifar10_tuning',
    overwrite=True
)

tuner.search_space_summary()

Search space summary
Default search space size: 5
num_layers (Int)
{'default': None, 'conditions': [], 'min_value': 1, 'max_value': 4, 'step': 1, 'sampling': 'linear'}
units_0 (Choice)
{'default': 64, 'conditions': [], 'values': [64, 128, 256, 512], 'ordered': True}
use_batchnorm (Boolean)
{'default': False, 'conditions': []}
dropout (Float)
{'default': 0.0, 'conditions': [], 'min_value': 0.0, 'max_value': 0.5, 'step': 0.1, 'sampling': 'linear'}
learning_rate (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.01, 'step': None, 'sampling': 'log'}


In [4]:
# Run the search
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)

tuner.search(
    X_train_flat, y_train,
    epochs=20,
    batch_size=256,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

# Show results
tuner.results_summary()

Trial 10 Complete [00h 00m 14s]
val_accuracy: 0.26010000705718994

Best val_accuracy So Far: 0.47510001063346863
Total elapsed time: 00h 03m 37s
Results summary
Results in tuner_results/cifar10_tuning
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 05 summary
Hyperparameters:
num_layers: 2
units_0: 256
use_batchnorm: False
dropout: 0.4
learning_rate: 0.0001073194620590324
units_1: 512
units_2: 256
units_3: 256
Score: 0.47510001063346863

Trial 02 summary
Hyperparameters:
num_layers: 3
units_0: 512
use_batchnorm: False
dropout: 0.0
learning_rate: 0.0007878587857071874
units_1: 512
units_2: 64
units_3: 256
Score: 0.47189998626708984

Trial 08 summary
Hyperparameters:
num_layers: 2
units_0: 64
use_batchnorm: False
dropout: 0.1
learning_rate: 0.0001729132516679919
units_1: 256
units_2: 256
units_3: 128
Score: 0.4700999855995178

Trial 07 summary
Hyperparameters:
num_layers: 4
units_0: 256
use_batchnorm: False
dropout: 0.2
learning_rate: 0.0003469634443643051
u

In [5]:
# Get best hyperparameters and model
best_hp = tuner.get_best_hyperparameters(1)[0]
print("\n=== Best Hyperparameters ===")
print(f"  Num layers: {best_hp.get('num_layers')}")
for i in range(best_hp.get('num_layers')):
    print(f"  Layer {i} units: {best_hp.get(f'units_{i}')}")
print(f"  BatchNorm: {best_hp.get('use_batchnorm')}")
print(f"  Dropout: {best_hp.get('dropout')}")
print(f"  Learning Rate: {best_hp.get('learning_rate'):.6f}")

# Build and evaluate the best model
best_model = tuner.get_best_models(1)[0]
loss, acc = best_model.evaluate(X_test_flat, y_test, verbose=0)
print(f"\nBest Model Test Accuracy: {acc:.4f}")


=== Best Hyperparameters ===
  Num layers: 2
  Layer 0 units: 256
  Layer 1 units: 512
  BatchNorm: False
  Dropout: 0.4
  Learning Rate: 0.000107


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))



Best Model Test Accuracy: 0.4858


## Bayesian Optimization Tuner
More efficient than random search — uses a probabilistic model to guide the search.

In [6]:
# Bayesian Optimization
bayes_tuner = kt.BayesianOptimization(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    directory='tuner_results',
    project_name='cifar10_bayes',
    overwrite=True
)

bayes_tuner.search(
    X_train_flat, y_train,
    epochs=20, batch_size=256,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

bayes_tuner.results_summary()
best_hp_bayes = bayes_tuner.get_best_hyperparameters(1)[0]
print(f"\nBayes Best LR: {best_hp_bayes.get('learning_rate'):.6f}")
print(f"Bayes Best Dropout: {best_hp_bayes.get('dropout')}")

Trial 10 Complete [00h 00m 25s]
val_accuracy: 0.4415999948978424

Best val_accuracy So Far: 0.47189998626708984
Total elapsed time: 00h 03m 35s
Results summary
Results in tuner_results/cifar10_bayes
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 03 summary
Hyperparameters:
num_layers: 2
units_0: 512
use_batchnorm: False
dropout: 0.30000000000000004
learning_rate: 0.00015313952522291152
units_1: 64
units_2: 256
Score: 0.47189998626708984

Trial 06 summary
Hyperparameters:
num_layers: 2
units_0: 64
use_batchnorm: False
dropout: 0.2
learning_rate: 0.0006137659553107151
units_1: 512
units_2: 256
units_3: 256
Score: 0.4650999903678894

Trial 01 summary
Hyperparameters:
num_layers: 2
units_0: 256
use_batchnorm: True
dropout: 0.1
learning_rate: 0.007912445575623542
units_1: 64
Score: 0.46239998936653137

Trial 08 summary
Hyperparameters:
num_layers: 1
units_0: 512
use_batchnorm: False
dropout: 0.2
learning_rate: 0.0008047741725190018
units_1: 128
units_2: 256
un

## Key Takeaways
- **Keras Tuner** automates hyperparameter search — no manual grid search
- **RandomSearch**: fast, simple, good baseline
- **BayesianOptimization**: smarter search, better for expensive models
- **Hyperband**: combines early stopping with random search for efficiency
- Define search space with `hp.Int()`, `hp.Float()`, `hp.Choice()`, `hp.Boolean()`
- Always use EarlyStopping callback during tuning to save time